In [0]:
%pip install great-expectations==0.18.21


  Using cached great_expectations-0.18.21-py3-none-any.whl.metadata (8.5 kB)
  Using cached altair-4.2.2-py3-none-any.whl.metadata (13 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached makefun-1.16.0-py2.py3-none-any.whl.metadata (2.9 kB)
  Using cached ruamel.yaml-0.17.40-py3-none-any.whl.metadata (19 kB)
  Using cached tzlocal-5.3.1-py3-none-any.whl.metadata (7.6 kB)
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached entrypoints-0.4-py3-none-any.whl.metadata (2.6 kB)
  Using cached toolz-1.1.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached ruamel_yaml_clib-0.2.15-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (3.5 kB)
Using cached great_expectations-0.18.21-py3-none-any.whl (5.4 MB)
Using cached altair-4.2.2-py3-none-any.whl (813 kB)
Using cached colorama-0.4.6-py2.py3-none-any.whl (25 kB)
Using cached makefun-1.16.0-py2.py3-none-any.w

# Gold Publish

Build candidate Gold tables from validated Silver, use GE to classify Gold row issues into `ERROR` and `WARNING`, quarantine invalid invoices, run final Spark integrity checks, then publish only valid Gold data.


In [0]:
from datetime import datetime, timezone
import uuid
import great_expectations as gx
from great_expectations.checkpoint import Checkpoint
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.functions import col, coalesce, when


In [0]:
STORAGE_ACCOUNT = "hantstorageaccount"
LAKEHOUSE_CONTAINER = "lakehouse"

CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "invoice"

SILVER_HEADER_PATH = dbutils.jobs.taskValues.get(
    taskKey="silver_ge", key="ge_valid_silver_header_path", debugValue=f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/validated/header/")
SILVER_LINES_PATH = dbutils.jobs.taskValues.get(
    taskKey="silver_ge", key="ge_valid_silver_lines_path", debugValue=f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/silver/validated/lines/")

GOLD_HEADER_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/validated/header/"
GOLD_LINES_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/validated/lines/"
GOLD_WARNING_HEADER_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/ge/warning/header/"
GOLD_WARNING_LINES_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/ge/warning/lines/"
GOLD_QUARANTINE_HEADER_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/ge/quarantine/header/"
GOLD_QUARANTINE_LINES_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/ge/quarantine/lines/"
GOLD_ISSUE_LOG_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/ge/issue_log/"
GOLD_RUN_SUMMARY_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/monitoring/run_summary/"
GOLD_SOURCE_SUMMARY_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/monitoring/source_publication_summary/"
GOLD_DQ_IMPACT_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/monitoring/dq_impact_summary/"
GOLD_KPI_RECON_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/gold/monitoring/kpi_reconciliation_summary/"

GE_RUNTIME_METRICS_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/monitoring/ge_runtime_metrics/"
GE_RUNTIME_METRICS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_ge_runtime_metrics"

GX_ROOT = "/dbfs/tmp/ge/gold_publish"
RUN_ID = str(uuid.uuid4())
ALLOWED_SOURCE_TYPES = ["csv", "json"]
ALLOWED_SHIP_MODES = ["FIRST CLASS",
                      "SECOND CLASS", "STANDARD CLASS", "SAME DAY"]
LINE_PASS_TOL = 0.01
LINE_ERROR_TOL = 0.05
HDR_PASS_TOL = 0.01
HDR_ERROR_TOL = 0.05

ISSUE_SCHEMA = T.StructType([
    T.StructField("layer", T.StringType(), False), T.StructField("dataset", T.StringType(), False), T.StructField(
        "rule_id", T.StringType(), False), T.StructField("severity", T.StringType(), False),
    T.StructField("SourceFile", T.StringType(), True), T.StructField("SourceType", T.StringType(), True), T.StructField(
        "InvoiceId", T.StringType(), True), T.StructField("LineNumber", T.StringType(), True),
    T.StructField("issue_type", T.StringType(), False), T.StructField("dq_reason", T.StringType(), False), T.StructField(
        "GoldRunId", T.StringType(), False), T.StructField("issue_ts", T.TimestampType(), False),
])

GE_RUNTIME_SCHEMA = T.StructType([
    T.StructField("layer", T.StringType(), False),
    T.StructField("suite_name", T.StringType(), False),
    T.StructField("run_id", T.StringType(), True),
    T.StructField("started_at", T.TimestampType(), False),
    T.StructField("ended_at", T.TimestampType(), False),
    T.StructField("runtime_seconds", T.DoubleType(), False),
    T.StructField("rows_evaluated", T.LongType(), True),
    T.StructField("rows_per_second", T.DoubleType(), True),
    T.StructField("evaluated_expectations", T.LongType(), True),
    T.StructField("successful_expectations", T.LongType(), True),
    T.StructField("failed_expectations", T.LongType(), True),
    T.StructField("expectation_success_percent", T.DoubleType(), True),
    T.StructField("validation_success", T.BooleanType(), True),
    T.StructField("recorded_at", T.TimestampType(), False),
])


HEADER_ORDER = ["InvoiceId", "OrderDate", "CustomerName", "ShipPostalCode", "ShipCity", "ShipState", "ShipCountry", "ShipMode", "BalanceDue", "SubTotal", "DiscountPercent",
                "DiscountAmount", "ShippingAmount", "InvoiceTotal", "OrderId", "SourceFile", "SourceType", "GoldRunId", "GoldLoadTimestamp", "dq_has_warning", "dq_warning_rules", "dq_warning_reasons"]
LINES_ORDER = ["InvoiceId", "LineNumber", "ProductName", "SubCategory", "Category", "ProductId", "Quantity", "UnitPrice", "ItemSubTotal",
               "SourceFile", "SourceType", "GoldRunId", "GoldLoadTimestamp", "dq_has_warning", "dq_warning_rules", "dq_warning_reasons"]
HEADER_ARITH = ["InvoiceId", "SourceFile", "SourceType", "SubTotal", "DiscountAmount", "ShippingAmount", "InvoiceTotal",
                "_line_count", "_sum_item_subtotal", "_calc_invoice_total", "_header_total_diff", "_line_to_subtotal_diff"]
LINES_ARITH = ["InvoiceId", "LineNumber", "SourceFile", "SourceType", "Quantity",
               "UnitPrice", "ItemSubTotal", "_calc_item_subtotal", "_line_subtotal_diff"]

print(SILVER_HEADER_PATH)
print(SILVER_LINES_PATH)
print(RUN_ID)

abfss://lakehouse@hantstorageaccount.dfs.core.windows.net/silver/validated/header/
abfss://lakehouse@hantstorageaccount.dfs.core.windows.net/silver/validated/lines/
91c89c32-1aef-4b2d-ac16-cf2597f4a108


In [0]:
def ensure_cols(df, cols, name):
    miss = [c for c in cols if c not in df.columns]
    if miss:
        raise RuntimeError(f"{name} missing columns: {miss}")


def reorder(df, cols):
    keep = [c for c in cols if c in df.columns]
    rem = [c for c in df.columns if c not in keep]
    return df.select(*keep, *rem)


def append_delta(df, path):
    df.write.format("delta").mode("append").save(path)


def with_run(df):
    return df.withColumn("GoldRunId", F.lit(RUN_ID)).withColumn("GoldLoadTimestamp", F.current_timestamp())


def empty_issue_df(): return spark.createDataFrame([], ISSUE_SCHEMA)


def gx_ctx(): return gx.get_context(context_root_dir=GX_ROOT)


def gx_ds(context, name):
    try:
        return context.sources.add_or_update_spark(name=name)
    except AttributeError:
        context.add_datasource(name, class_name="Datasource", execution_engine={"class_name": "SparkDFExecutionEngine"}, data_connectors={
                           "runtime_data_connector": {"class_name": "RuntimeDataConnector", "batch_identifiers": ["default_identifier_name"]}})
        return context.get_datasource(name)


def batch_req(datasource, name, df): return datasource.add_dataframe_asset(
    name=name).build_batch_request(dataframe=df)


def add_ckpt(context, name, req, suite):
    ck = Checkpoint(
        name=name, 
        data_context=context, 
        validations=[
            {
                "batch_request": req, 
                "expectation_suite_name": suite
            }
        ], 
        action_list=[
            {
                "name": "store_validation_result",
                "action": {"class_name": "StoreValidationResultAction"},
            },
            {
                "name": "update_data_docs",
                "action": {"class_name": "UpdateDataDocsAction"},
            },
        ],
        run_name_template="%Y%m%dT%H%M%S_gold_publish"
    )
    context.add_or_update_checkpoint(checkpoint=ck)
    return ck


def first_val(res): return res.run_results[list(
    res.run_results.keys())[0]]["validation_result"]


def summary(validation, s):
    st = validation.get("statistics", {}) or {}
    return {"suite": s, "success": validation.get("success"), "evaluated": st.get("evaluated_expectations", 0), "successful": st.get("successful_expectations", 0), "failed": st.get("unsuccessful_expectations", 0)}


def show_summary(s):
    print("="*60)
    print(f"{s['suite']}: {'PASSED' if s['success'] else 'FAILED'}")
    print(
        f"Evaluated={s['evaluated']} Passed={s['successful']} Failed={s['failed']}")


def rf(keys): return {"result_format": "COMPLETE",
                      "unexpected_index_column_names": keys, "return_unexpected_index_query": True}


def meta(rule, sev, datasource, itype, reason, keys): return {
    "rule_id": rule, "severity": sev, "dataset": datasource, "issue_type": itype, "dq_reason": reason, "key_columns": keys}


def issue_key(item, name):
    if hasattr(item, "asDict"):
        item = item.asDict(recursive=True)
    if not isinstance(item, dict):
        return None
    value = item.get(name)
    nested_index = item.get("unexpected_index")
    if value is None and isinstance(nested_index, dict):
        value = nested_index.get(name)
    return None if value is None else str(value)


def issue_df(results):
    rows = []
    for vr in results:
        for r in vr.get("results", []):
            if r.get("success", True):
                continue
            m = (r.get("expectation_config", {}) or {}).get("meta", {}) or {}
            if not m.get("rule_id") or not m.get("severity"):
                continue
            payload = (r.get("result", {}) or {})
            idx = (payload.get("unexpected_index_list") or payload.get(
                "partial_unexpected_index_list") or payload.get("unexpected_rows") or [None])
            for item in idx:
                rows.append({"layer": "gold", "dataset": m.get("dataset"), "rule_id": m.get("rule_id"), "severity": m.get("severity"), "SourceFile": issue_key(item, "SourceFile"), "SourceType": issue_key(item, "SourceType"), "InvoiceId": issue_key(
                    item, "InvoiceId"), "LineNumber": issue_key(item, "LineNumber"), "issue_type": m.get("issue_type"), "dq_reason": m.get("dq_reason"), "GoldRunId": RUN_ID, "issue_ts": datetime.now(timezone.utc)})
    return empty_issue_df() if not rows else spark.createDataFrame(rows, ISSUE_SCHEMA).dropDuplicates()


def failed_expectation_rows(results):
    rows = []
    for vr in results:
        suite = ((vr.get("meta", {}) or {}).get("expectation_suite_name"))
        for r in vr.get("results", []):
            if r.get("success", True):
                continue
            cfg = r.get("expectation_config", {}) or {}
            m = cfg.get("meta", {}) or {}
            res = r.get("result", {}) or {}
            rows.append({"suite": suite, "dataset": m.get("dataset"), "rule_id": m.get("rule_id"), "severity": (m.get("severity") or "UNSPECIFIED").upper(), "expectation_type": cfg.get("expectation_type"), "unexpected_count": res.get("unexpected_count"), "dq_reason": m.get("dq_reason")})
    return rows


def raise_for_error_expectations(failed_expectations):
    warning_count = sum(1 for x in failed_expectations if x.get("severity") == "WARNING")
    error_expectations = [x for x in failed_expectations if x.get("severity") == "ERROR"]
    if warning_count:
        print(f"Gold WARNING expectation failure(s): {warning_count}. Continuing without RuntimeError.")
    if not error_expectations:
        return
    rules = sorted({f"{x.get('dataset')}.{x.get('rule_id')}" for x in error_expectations})
    preview = ", ".join(rules[:10])
    if len(rules) > 10:
        preview += f", ... +{len(rules) - 10} more"
    raise RuntimeError(f"Gold STOP failure: {len(error_expectations)} ERROR expectation(s) failed: {preview}")


def line_arith(df):
    return df.withColumn("_calc_item_subtotal", when(col("Quantity").isNotNull() & col("UnitPrice").isNotNull(), (col("Quantity")*col("UnitPrice")).cast("decimal(18,2)")).otherwise(None)).withColumn("_line_subtotal_diff", when(col("ItemSubTotal").isNotNull() & col("_calc_item_subtotal").isNotNull(), (col("ItemSubTotal")-col("_calc_item_subtotal")).cast("decimal(18,2)")).otherwise(None))


def hdr_arith(h, l):
    roll = line_arith(l).groupBy("InvoiceId").agg(F.count("LineNumber").cast("long").alias(
        "_line_count"), F.sum("ItemSubTotal").cast("decimal(18,2)").alias("_sum_item_subtotal"))
    return h.join(roll, on="InvoiceId", how="left").withColumn("_calc_invoice_total", when(col("SubTotal").isNotNull(), (col("SubTotal")-coalesce(col("DiscountAmount"), F.lit(0).cast("decimal(18,2)"))+coalesce(col("ShippingAmount"), F.lit(0).cast("decimal(18,2)"))).cast("decimal(18,2)")).otherwise(None)).withColumn("_header_total_diff", when(col("InvoiceTotal").isNotNull() & col("_calc_invoice_total").isNotNull(), (col("InvoiceTotal")-col("_calc_invoice_total")).cast("decimal(18,2)")).otherwise(None)).withColumn("_line_to_subtotal_diff", when(col("SubTotal").isNotNull() & col("_sum_item_subtotal").isNotNull(), (col("SubTotal")-col("_sum_item_subtotal")).cast("decimal(18,2)")).otherwise(None))

# Runtime metrics recording helper functions
def validation_statistics(validation_result):
    stats = validation_result.get("statistics", {}) or {}
    evaluated = int(stats.get("evaluated_expectations", 0) or 0)
    successful = int(stats.get("successful_expectations", 0) or 0)
    failed = int(stats.get("unsuccessful_expectations", 0) or 0)
    success_percent = float(successful / evaluated) if evaluated else None
    return evaluated, successful, failed, success_percent


def write_ge_runtime_metric(layer, suite_name, run_id, started_at, ended_at, rows_evaluated, validation_result):
    runtime_seconds = (ended_at - started_at).total_seconds()
    evaluated, successful, failed, success_percent = validation_statistics(validation_result)
    rows_per_second = float(rows_evaluated / runtime_seconds) if rows_evaluated and runtime_seconds > 0 else None

    row = [(
        layer,
        suite_name,
        run_id,
        started_at,
        ended_at,
        float(runtime_seconds),
        int(rows_evaluated) if rows_evaluated is not None else None,
        rows_per_second,
        evaluated,
        successful,
        failed,
        success_percent,
        bool(validation_result.get("success")),
        datetime.now(timezone.utc),
    )]

    runtime_df = spark.createDataFrame(row, GE_RUNTIME_SCHEMA)
    runtime_df.write.format("delta").mode("append").save(GE_RUNTIME_METRICS_PATH)

    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")
    spark.sql(f'''
        CREATE TABLE IF NOT EXISTS {GE_RUNTIME_METRICS_TABLE}
        USING DELTA
        LOCATION "{GE_RUNTIME_METRICS_PATH}"
    ''')



In [0]:
# runtime metrics helper functions
def first_validation_result(checkpoint_result):
    run_results = checkpoint_result.run_results
    return run_results[list(run_results.keys())[0]]["validation_result"]


def validation_statistics(validation_result):
    stats = validation_result.get("statistics", {}) or {}
    evaluated = int(stats.get("evaluated_expectations", 0) or 0)
    successful = int(stats.get("successful_expectations", 0) or 0)
    failed = int(stats.get("unsuccessful_expectations", 0) or 0)
    success_percent = float(successful / evaluated) if evaluated else None
    return evaluated, successful, failed, success_percent


def write_ge_runtime_metric(layer, suite_name, run_id, started_at, ended_at, rows_evaluated, validation_result):
    runtime_seconds = (ended_at - started_at).total_seconds()
    evaluated, successful, failed, success_percent = validation_statistics(validation_result)
    rows_per_second = float(rows_evaluated / runtime_seconds) if rows_evaluated and runtime_seconds > 0 else None

    row = [(
        layer,
        suite_name,
        run_id,
        started_at,
        ended_at,
        float(runtime_seconds),
        int(rows_evaluated) if rows_evaluated is not None else None,
        rows_per_second,
        evaluated,
        successful,
        failed,
        success_percent,
        bool(validation_result.get("success")),
        datetime.now(timezone.utc),
    )]

    runtime_df = spark.createDataFrame(row, GE_RUNTIME_SCHEMA)
    runtime_df.write.format("delta").mode("append").save(GE_RUNTIME_METRICS_PATH)

    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")
    spark.sql(f'''
        CREATE TABLE IF NOT EXISTS {GE_RUNTIME_METRICS_TABLE}
        USING DELTA
        LOCATION "{GE_RUNTIME_METRICS_PATH}"
    ''')


In [0]:
silver_header_df = spark.read.format("delta").load(SILVER_HEADER_PATH)
silver_lines_df = spark.read.format("delta").load(SILVER_LINES_PATH)

if silver_header_df.rdd.isEmpty() or silver_lines_df.rdd.isEmpty():
    raise RuntimeError("Gold STOP failure: validated Silver inputs are empty.")

ensure_cols(silver_header_df, ["_source_file", "source_type", "InvoiceId", "OrderDate",
            "CustomerName", "ShipMode", "SubTotal", "InvoiceTotal"], "silver_header_df")
ensure_cols(silver_lines_df, ["_source_file", "source_type", "InvoiceId", "LineNumber",
            "ProductName", "Quantity", "UnitPrice", "ItemSubTotal"], "silver_lines_df")
gold_header_df = reorder(silver_header_df.withColumn("ShipMode", when(col("ShipMode").isNotNull(), F.upper(F.trim(col("ShipMode")))).otherwise(
    None)).withColumnRenamed("_source_file", "SourceFile").withColumnRenamed("source_type", "SourceType").transform(with_run), HEADER_ORDER)
gold_lines_df = reorder(silver_lines_df.withColumnRenamed("_source_file", "SourceFile").withColumnRenamed(
    "source_type", "SourceType").transform(with_run), LINES_ORDER)


def run_header(df, lbl):
    context = gx_ctx()
    datasource = gx_ds(context, f"gold_header_{lbl}")
    req = batch_req(datasource, f"gold_header_asset_{lbl}", df)
    suite = f"gold_header_suite_{lbl}"
    context.add_or_update_expectation_suite(expectation_suite_name=suite)
    validation = context.get_validator(batch_request=req, expectation_suite_name=suite)
    header_keys = ["InvoiceId", "SourceFile", "SourceType"]
    for col_name in ["InvoiceId", "OrderDate", "CustomerName", "ShipMode", "SubTotal", "InvoiceTotal", "SourceFile", "SourceType"]:
        validation.expect_column_values_to_not_be_null(col_name, result_format=rf(header_keys), meta=meta(
            f"gold_header_{col_name.lower()}_present", "ERROR", "header", "contract", f"{col_name} is missing in Gold header", header_keys))
    validation.expect_column_values_to_be_in_set("SourceType", ALLOWED_SOURCE_TYPES, result_format=rf(header_keys), meta=meta(
        "gold_header_source_type_domain", "ERROR", "header", "contract", "SourceType is outside the allowed domain", header_keys))
    validation.expect_column_values_to_be_in_set("ShipMode", ALLOWED_SHIP_MODES, result_format=rf(header_keys), meta=meta(
        "gold_header_ship_mode_domain", "ERROR", "header", "contract", "ShipMode is outside the allowed domain", header_keys))
    for col_name in ["SubTotal", "InvoiceTotal", "BalanceDue", "DiscountAmount", "ShippingAmount"]:
        validation.expect_column_values_to_be_between(col_name, min_value=0, max_value=None, mostly=0.99, result_format=rf(header_keys), meta=meta(
            f"gold_header_{col_name.lower()}_non_negative", "ERROR", "header", "contract", f"{col_name} is negative in Gold header", header_keys))
    validation.expect_column_values_to_be_between("DiscountPercent", min_value=0, max_value=1, mostly=0.99, result_format=rf(
        header_keys), meta=meta("gold_header_discount_percent_range", "ERROR", "header", "contract", "DiscountPercent is outside [0,1]", header_keys))
    for col_name in ["ShipPostalCode", "ShipCity", "ShipState", "ShipCountry"]:
        validation.expect_column_values_to_not_be_null(col_name, result_format=rf(header_keys), meta=meta(
            f"gold_header_{col_name.lower()}_warning", "WARNING", "header", "contract", f"{col_name} is missing in Gold header", header_keys))
    validation.save_expectation_suite(discard_failed_expectations=False)
    add_ckpt(context, f"gold_header_ck_{lbl}", req, suite)
    started_at = datetime.now(timezone.utc)
    checkpoint_result = context.run_checkpoint(checkpoint_name=f"gold_header_ck_{lbl}", runtime_configuration={"result_format": rf(header_keys)})
    ended_at = datetime.now(timezone.utc)
    validation_result = first_validation_result(checkpoint_result)
    write_ge_runtime_metric(
        layer="gold",
        suite_name=suite,
        run_id=RUN_ID,
        started_at=started_at,
        ended_at=ended_at,
        rows_evaluated=df.count(),
        validation_result=validation_result
    )
    return checkpoint_result, suite


def run_lines(df, lbl):
    context = gx_ctx()
    datasource = gx_ds(context, f"gold_lines_{lbl}")
    req = batch_req(datasource, f"gold_lines_asset_{lbl}", df)
    suite = f"gold_lines_suite_{lbl}"
    context.add_or_update_expectation_suite(expectation_suite_name=suite)
    validation = context.get_validator(batch_request=req, expectation_suite_name=suite)
    line_keys = ["InvoiceId", "LineNumber", "SourceFile", "SourceType"]
    for c in ["InvoiceId", "LineNumber", "ProductName", "Quantity", "UnitPrice", "ItemSubTotal", "SourceFile", "SourceType"]:
        validation.expect_column_values_to_not_be_null(c, result_format=rf(line_keys), meta=meta(
            f"gold_lines_{c.lower()}_present", "ERROR", "lines", "contract", f"{c} is missing in Gold lines", line_keys))
    validation.expect_column_values_to_be_in_set("SourceType", ALLOWED_SOURCE_TYPES, result_format=rf(line_keys), meta=meta(
        "gold_lines_source_type_domain", "ERROR", "lines", "contract", "SourceType is outside the allowed domain", line_keys))
    validation.expect_column_values_to_be_between("Quantity", min_value=1, max_value=None, mostly=0.99, result_format=rf(
        line_keys), meta=meta("gold_lines_quantity_positive", "ERROR", "lines", "contract", "Quantity must be greater than zero", line_keys))
    for c in ["UnitPrice", "ItemSubTotal"]:
        validation.expect_column_values_to_be_between(c, min_value=0, max_value=None, mostly=0.99, result_format=rf(line_keys), meta=meta(
            f"gold_lines_{c.lower()}_non_negative", "ERROR", "lines", "contract", f"{c} is negative in Gold lines", line_keys))
    for c in ["Category", "SubCategory", "ProductId"]:
        validation.expect_column_values_to_not_be_null(c, result_format=rf(line_keys), meta=meta(
            f"gold_lines_{c.lower()}_warning", "WARNING", "lines", "contract", f"{c} is missing in Gold lines", line_keys))
    validation.save_expectation_suite(discard_failed_expectations=False)
    add_ckpt(context, f"gold_lines_ck_{lbl}", req, suite)
    started_at = datetime.now(timezone.utc)
    checkpoint_result = context.run_checkpoint(checkpoint_name=f"gold_lines_ck_{lbl}", runtime_configuration={"result_format": rf(line_keys)})
    ended_at = datetime.now(timezone.utc)
    validation_result = first_validation_result(checkpoint_result)
    write_ge_runtime_metric(
        layer="gold",
        suite_name=suite,
        run_id=RUN_ID,
        started_at=started_at,
        ended_at=ended_at,
        rows_evaluated=df.count(),
        validation_result=validation_result
    )
    return checkpoint_result, suite


def run_hdr_arith(h, l, lbl):
    context = gx_ctx()
    datasource = gx_ds(context, f"gold_hdr_arith_{lbl}")
    hdr_arith_df = hdr_arith(h, l)
    req = batch_req(datasource, f"gold_hdr_arith_asset_{lbl}", hdr_arith_df.select(*HEADER_ARITH))
    suite = f"gold_hdr_arith_suite_{lbl}"
    context.add_or_update_expectation_suite(expectation_suite_name=suite)
    validation = context.get_validator(batch_request=req, expectation_suite_name=suite)
    k = ["InvoiceId", "SourceFile", "SourceType"]
    validation.expect_column_values_to_be_between("_header_total_diff", min_value=-HDR_ERROR_TOL, max_value=HDR_ERROR_TOL, result_format=rf(k), meta=meta(
        "gold_header_arith_total_error", "ERROR", "header", "header_total_diff", "InvoiceTotal exceeds the Gold reconciliation error band", k))
    validation.expect_column_values_to_be_between("_line_to_subtotal_diff", min_value=-HDR_ERROR_TOL, max_value=HDR_ERROR_TOL, result_format=rf(k), meta=meta(
        "gold_header_arith_subtotal_error", "ERROR", "header", "line_to_subtotal_diff", "SubTotal exceeds the Gold reconciliation error band", k))
    validation.expect_column_values_to_be_between("_header_total_diff", min_value=-HDR_PASS_TOL, max_value=HDR_PASS_TOL, result_format=rf(k), meta=meta(
        "gold_header_arith_total_warning", "WARNING", "header", "header_total_diff", "InvoiceTotal exceeds the strict Gold reconciliation tolerance", k))
    validation.expect_column_values_to_be_between("_line_to_subtotal_diff", min_value=-HDR_PASS_TOL, max_value=HDR_PASS_TOL, result_format=rf(k), meta=meta(
        "gold_header_arith_subtotal_warning", "WARNING", "header", "line_to_subtotal_diff", "SubTotal exceeds the strict Gold reconciliation tolerance", k))
    validation.save_expectation_suite(discard_failed_expectations=False)
    add_ckpt(context, f"gold_hdr_arith_ck_{lbl}", req, suite)
    started_at = datetime.now(timezone.utc)
    checkpoint_result = context.run_checkpoint(checkpoint_name=f"gold_hdr_arith_ck_{lbl}", runtime_configuration={"result_format": rf(k)})
    ended_at = datetime.now(timezone.utc)
    validation_result = first_validation_result(checkpoint_result)
    write_ge_runtime_metric(
        layer="gold",
        suite_name=suite,
        run_id=RUN_ID,
        started_at=started_at,
        ended_at=ended_at,
        rows_evaluated=hdr_arith_df.count(),
        validation_result=validation_result
    )
    return checkpoint_result, suite


def run_line_arith(l, lbl):
    context = gx_ctx()
    line_arith_df = line_arith(l)
    datasource = gx_ds(context, f"gold_line_arith_{lbl}")
    req = batch_req(datasource, f"gold_line_arith_asset_{lbl}", line_arith_df.select(*LINES_ARITH))
    suite = f"gold_line_arith_suite_{lbl}"
    context.add_or_update_expectation_suite(expectation_suite_name=suite)
    validation = context.get_validator(batch_request=req, expectation_suite_name=suite)
    k = ["InvoiceId", "LineNumber", "SourceFile", "SourceType"]
    validation.expect_column_values_to_be_between("_line_subtotal_diff", min_value=-LINE_ERROR_TOL, max_value=LINE_ERROR_TOL, result_format=rf(
        k), meta=meta("gold_line_arith_error", "ERROR", "lines", "line_subtotal_diff", "Line subtotal exceeds the Gold reconciliation error band", k))
    validation.expect_column_values_to_be_between("_line_subtotal_diff", min_value=-LINE_PASS_TOL, max_value=LINE_PASS_TOL, result_format=rf(k), meta=meta(
        "gold_line_arith_warning", "WARNING", "lines", "line_subtotal_diff", "Line subtotal exceeds the strict Gold reconciliation tolerance", k))
    validation.save_expectation_suite(discard_failed_expectations=False)
    add_ckpt(context, f"gold_line_arith_ck_{lbl}", req, suite)
    started_at = datetime.now(timezone.utc)
    checkpoint_result = context.run_checkpoint(checkpoint_name=f"gold_line_arith_ck_{lbl}", runtime_configuration={"result_format": rf(k)})
    ended_at = datetime.now(timezone.utc)
    validation_result = first_validation_result(checkpoint_result)
    write_ge_runtime_metric(
        layer="gold",
        suite_name=suite,
        run_id=RUN_ID,
        started_at=started_at,
        ended_at=ended_at,
        rows_evaluated=line_arith_df.count(),
        validation_result=validation_result
    )
    return checkpoint_result, suite

In [0]:
res_h, s_h = run_header(gold_header_df, "pre")
res_l, s_l = run_lines(gold_lines_df, "pre")
res_ha, s_ha = run_hdr_arith(gold_header_df, gold_lines_df, "pre")
res_la, s_la = run_line_arith(gold_lines_df, "pre")
vals = [first_val(res_h), first_val(res_l),
        first_val(res_ha), first_val(res_la)]
for s in [summary(vals[0], s_h), summary(vals[1], s_l), summary(vals[2], s_ha), summary(vals[3], s_la)]:
    show_summary(s)
issues = issue_df(vals)
failed_expectations = failed_expectation_rows(vals)
err_ids = issues.filter((col("severity") == "ERROR") & col(
    "InvoiceId").isNotNull()).select("InvoiceId").distinct()
warn_ids = issues.filter((col("severity") == "WARNING") & col("InvoiceId").isNotNull(
)).select("InvoiceId").distinct().join(err_ids, on="InvoiceId", how="left_anti")
warn_roll = issues.filter((col("severity") == "WARNING") & col("InvoiceId").isNotNull()).select("InvoiceId", "rule_id", "dq_reason").groupBy(
    "InvoiceId").agg(F.collect_set("rule_id").alias("dq_warning_rules"), F.collect_set("dq_reason").alias("dq_warning_reasons"))
valid_h = gold_header_df.join(err_ids, on="InvoiceId", how="left_anti").drop("dq_warning_rules", "dq_warning_reasons", "dq_has_warning").join(warn_roll, on="InvoiceId", how="left").withColumn(
    "dq_has_warning", F.when(col("dq_warning_rules").isNotNull(), F.lit(True)).otherwise(F.lit(False)))
valid_l = gold_lines_df.join(err_ids, on="InvoiceId", how="left_anti").drop("dq_warning_rules", "dq_warning_reasons", "dq_has_warning").join(warn_roll, on="InvoiceId", how="left").withColumn(
    "dq_has_warning", F.when(col("dq_warning_rules").isNotNull(), F.lit(True)).otherwise(F.lit(False)))
warn_h = gold_header_df.join(warn_ids, on="InvoiceId", how="inner").drop("dq_warning_rules", "dq_warning_reasons", "dq_has_warning").join(
    warn_roll, on="InvoiceId", how="left").withColumn("dq_has_warning", F.lit(True))
warn_l = gold_lines_df.join(warn_ids, on="InvoiceId", how="inner").drop("dq_warning_rules", "dq_warning_reasons", "dq_has_warning").join(
    warn_roll, on="InvoiceId", how="left").withColumn("dq_has_warning", F.lit(True))
quar_h = gold_header_df.join(err_ids, on="InvoiceId", how="inner")
quar_l = gold_lines_df.join(err_ids, on="InvoiceId", how="inner")
hdr_dup = valid_h.groupBy("InvoiceId").count().filter(col("count") > 1).count()
line_dup = valid_l.groupBy("InvoiceId", "LineNumber").count().filter(
    col("count") > 1).count()
orphan = valid_l.join(valid_h.select("InvoiceId").distinct(),
                      on="InvoiceId", how="left_anti").count()
headless = valid_h.join(valid_l.select(
    "InvoiceId").distinct(), on="InvoiceId", how="left_anti").count()
if hdr_dup > 0 or line_dup > 0 or orphan > 0 or headless > 0:
    raise RuntimeError(
        f"Gold STOP failure after GE routing: header_duplicate_count={hdr_dup}, line_duplicate_count={line_dup}, orphan_line_count={orphan}, header_without_lines_count={headless}")
display(issues.limit(50))
raise_for_error_expectations(failed_expectations)

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/13 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/13 [00:00<?, ?it/s]

gold_header_suite_pre: FAILED
Evaluated=20 Passed=16 Failed=4
gold_lines_suite_pre: PASSED
Evaluated=15 Passed=15 Failed=0
gold_hdr_arith_suite_pre: PASSED
Evaluated=2 Passed=2 Failed=0
gold_line_arith_suite_pre: FAILED
Evaluated=1 Passed=0 Failed=1


layer,dataset,rule_id,severity,SourceFile,SourceType,InvoiceId,LineNumber,issue_type,dq_reason,GoldRunId,issue_ts
gold,header,gold_header_shippostalcode_warning,WARNING,null,null,null,null,contract,ShipPostalCode is missing in Gold header,91c89c32-1aef-4b2d-ac16-cf2597f4a108,2026-04-23T19:52:50.161611Z
gold,header,gold_header_shipcity_warning,WARNING,null,null,null,null,contract,ShipCity is missing in Gold header,91c89c32-1aef-4b2d-ac16-cf2597f4a108,2026-04-23T19:52:50.161632Z
gold,header,gold_header_shipstate_warning,WARNING,null,null,null,null,contract,ShipState is missing in Gold header,91c89c32-1aef-4b2d-ac16-cf2597f4a108,2026-04-23T19:52:50.161647Z
gold,lines,gold_line_arith_warning,WARNING,null,null,null,null,line_subtotal_diff,Line subtotal exceeds the strict Gold reconciliation tolerance,91c89c32-1aef-4b2d-ac16-cf2597f4a108,2026-04-23T19:52:50.161696Z
gold,header,gold_header_shipcountry_warning,WARNING,null,null,null,null,contract,ShipCountry is missing in Gold header,91c89c32-1aef-4b2d-ac16-cf2597f4a108,2026-04-23T19:52:50.161661Z


Gold WARNING expectation failure(s): 5. Continuing without RuntimeError.


In [0]:
roll = (
    valid_l
    .groupBy("InvoiceId")
    .agg(
        F.count("LineNumber").cast("long").alias("PublishedLineCount"),
        F.sum("ItemSubTotal").cast("decimal(18,2)").alias(
            "PublishedLinesSubTotal"),
    )
)

kpi = (
    valid_h
    .join(roll, on="InvoiceId", how="left")
    .withColumn(
        "HeaderVsLinesSubTotalDiff",
        (
            F.coalesce(col("SubTotal"), F.lit(0).cast("decimal(18,2)"))
            - F.coalesce(col("PublishedLinesSubTotal"),
                         F.lit(0).cast("decimal(18,2)"))
        ).cast("decimal(18,2)"),
    )
    .withColumn(
        "HeaderVsLinesTotalDiff",
        (
            F.coalesce(col("InvoiceTotal"), F.lit(0).cast("decimal(18,2)"))
            - (
                F.coalesce(col("PublishedLinesSubTotal"),
                           F.lit(0).cast("decimal(18,2)"))
                - F.coalesce(col("DiscountAmount"),
                             F.lit(0).cast("decimal(18,2)"))
                + F.coalesce(col("ShippingAmount"),
                             F.lit(0).cast("decimal(18,2)"))
            )
        ).cast("decimal(18,2)"),
    )
    .withColumn("GoldRunId", F.lit(RUN_ID))
    .withColumn("GoldLoadTimestamp", F.current_timestamp())
)

run_summary = (
    valid_h
    .agg(
        F.countDistinct("InvoiceId").cast(
            "long").alias("PublishedInvoiceCount"),
        F.sum("InvoiceTotal").cast("decimal(18,2)").alias(
            "PublishedInvoiceRevenue"),
        F.sum("SubTotal").cast("decimal(18,2)").alias("PublishedSubTotal"),
        F.min("OrderDate").alias("MinOrderDate"),
        F.max("OrderDate").alias("MaxOrderDate"),
    )
    .crossJoin(
        valid_l.agg(F.count(F.lit(1)).cast("long").alias("PublishedLineCount"))
    )
    .withColumn(
        "WarningInvoiceCount",
        F.lit(warn_h.select("InvoiceId").distinct().count()).cast("long"),
    )
    .withColumn(
        "QuarantineInvoiceCount",
        F.lit(quar_h.select("InvoiceId").distinct().count()).cast("long"),
    )
    .withColumn("WarningLineCount", F.lit(warn_l.count()).cast("long"))
    .withColumn("QuarantineLineCount", F.lit(quar_l.count()).cast("long"))
    .withColumn("HeaderDuplicateCount", F.lit(hdr_dup).cast("long"))
    .withColumn("LineDuplicateCount", F.lit(line_dup).cast("long"))
    .withColumn("OrphanLineCount", F.lit(orphan).cast("long"))
    .withColumn("HeaderWithoutLinesCount", F.lit(headless).cast("long"))
    .withColumn(
        "MaxAbsHeaderVsLinesSubTotalDiff",
        F.lit(
            kpi.agg(F.max(F.abs(col("HeaderVsLinesSubTotalDiff")))).first()[0]),
    )
    .withColumn(
        "MaxAbsHeaderVsLinesTotalDiff",
        F.lit(kpi.agg(F.max(F.abs(col("HeaderVsLinesTotalDiff")))).first()[0]),
    )
    .withColumn("GoldRunId", F.lit(RUN_ID))
    .withColumn("GoldLoadTimestamp", F.current_timestamp())
)

source_summary = (
    valid_h
    .groupBy("SourceType")
    .agg(
        F.countDistinct("InvoiceId").cast(
            "long").alias("PublishedInvoiceCount"),
        F.sum("InvoiceTotal").cast("decimal(18,2)").alias("PublishedRevenue"),
        F.min("OrderDate").alias("MinOrderDate"),
        F.max("OrderDate").alias("MaxOrderDate"),
    )
    .withColumn("GoldRunId", F.lit(RUN_ID))
    .withColumn("GoldLoadTimestamp", F.current_timestamp())
)

if issues.rdd.isEmpty():
    dq_impact = spark.createDataFrame(
        [],
        "SourceType string, ImpactType string, AffectedInvoices long, AffectedLines long, "
        "AffectedRules long, GoldRunId string, GoldLoadTimestamp timestamp",
    )
else:
    dq_impact = (
        issues
        .withColumn("SourceType", F.coalesce(col("SourceType"), F.lit("unknown")))
        .withColumn("ImpactType", F.concat(F.lower(col("severity")), F.lit("_"), col("dataset")))
        .withColumn(
            "_line_key",
            F.when(
                col("dataset") == "lines",
                F.concat_ws(
                    "::",
                    F.coalesce(col("InvoiceId"), F.lit("")),
                    F.coalesce(col("LineNumber"), F.lit("")),
                ),
            ),
        )
        .groupBy("SourceType", "ImpactType")
        .agg(
            F.countDistinct("InvoiceId").cast(
                "long").alias("AffectedInvoices"),
            F.countDistinct("_line_key").cast("long").alias("AffectedLines"),
            F.countDistinct("rule_id").cast("long").alias("AffectedRules"),
        )
        .withColumn("GoldRunId", F.lit(RUN_ID))
        .withColumn("GoldLoadTimestamp", F.current_timestamp())
    )

valid_h.write.format("delta").mode("overwrite").save(GOLD_HEADER_PATH)
valid_l.write.format("delta").mode("overwrite").save(GOLD_LINES_PATH)
warn_h.write.format("delta").mode("overwrite").save(GOLD_WARNING_HEADER_PATH)
warn_l.write.format("delta").mode("overwrite").save(GOLD_WARNING_LINES_PATH)
quar_h.write.format("delta").mode(
    "overwrite").save(GOLD_QUARANTINE_HEADER_PATH)
quar_l.write.format("delta").mode("overwrite").save(GOLD_QUARANTINE_LINES_PATH)
issues.write.format("delta").mode("append").save(GOLD_ISSUE_LOG_PATH)

append_delta(run_summary, GOLD_RUN_SUMMARY_PATH)
append_delta(source_summary, GOLD_SOURCE_SUMMARY_PATH)
append_delta(dq_impact, GOLD_DQ_IMPACT_PATH)
append_delta(kpi, GOLD_KPI_RECON_PATH)

In [0]:
CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "invoice"

BATCH_GOLD_VALIDATED_HEADER_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_gold_validated_header"
BATCH_GOLD_VALIDATED_LINES_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_gold_validated_lines"
BATCH_GOLD_WARNING_HEADER_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_gold_warning_header"
BATCH_GOLD_WARNING_LINES_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_gold_warning_lines"
BATCH_GOLD_QUARANTINE_HEADER_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_gold_quarantine_header"
BATCH_GOLD_QUARANTINE_LINES_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_gold_quarantine_lines"
BATCH_GOLD_GE_ISSUE_LOG_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_gold_ge_issue_log"
BATCH_GOLD_RUN_SUMMARY_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_gold_run_summary"
BATCH_GOLD_SOURCE_SUMMARY_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_gold_source_publication_summary"
BATCH_GOLD_DQ_IMPACT_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_gold_dq_impact_summary"
BATCH_GOLD_KPI_RECON_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.batch_gold_kpi_reconciliation_summary"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_GOLD_VALIDATED_HEADER_TABLE}
USING DELTA
LOCATION "{GOLD_HEADER_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_GOLD_VALIDATED_LINES_TABLE}
USING DELTA
LOCATION "{GOLD_LINES_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_GOLD_WARNING_HEADER_TABLE}
USING DELTA
LOCATION "{GOLD_WARNING_HEADER_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_GOLD_WARNING_LINES_TABLE}
USING DELTA
LOCATION "{GOLD_WARNING_LINES_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_GOLD_QUARANTINE_HEADER_TABLE}
USING DELTA
LOCATION "{GOLD_QUARANTINE_HEADER_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_GOLD_QUARANTINE_LINES_TABLE}
USING DELTA
LOCATION "{GOLD_QUARANTINE_LINES_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_GOLD_GE_ISSUE_LOG_TABLE}
USING DELTA
LOCATION "{GOLD_ISSUE_LOG_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_GOLD_RUN_SUMMARY_TABLE}
USING DELTA
LOCATION "{GOLD_RUN_SUMMARY_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_GOLD_SOURCE_SUMMARY_TABLE}
USING DELTA
LOCATION "{GOLD_SOURCE_SUMMARY_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_GOLD_DQ_IMPACT_TABLE}
USING DELTA
LOCATION "{GOLD_DQ_IMPACT_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {BATCH_GOLD_KPI_RECON_TABLE}
USING DELTA
LOCATION "{GOLD_KPI_RECON_PATH}"
''')

display(spark.sql(f"DESCRIBE DETAIL {BATCH_GOLD_RUN_SUMMARY_TABLE}"))
display(spark.sql(f"DESCRIBE DETAIL {BATCH_GOLD_KPI_RECON_TABLE}"))


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,35b38f66-15eb-485a-a07c-9859c9f81151,hant-catalog.invoice.batch_gold_run_summary,null,abfss://lakehouse@hantstorageaccount.dfs.core.windows.net/gold/monitoring/run_summary,2026-04-23T19:53:12.521Z,2026-04-23T19:53:13Z,List(),List(),1,6042,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,d924dc20-fa9e-49f2-8187-82bd1bf9757d,hant-catalog.invoice.batch_gold_kpi_reconciliation_summary,null,abfss://lakehouse@hantstorageaccount.dfs.core.windows.net/gold/monitoring/kpi_reconciliation_summary,2026-04-23T19:53:17.789Z,2026-04-23T19:53:19Z,List(),List(),1,105255,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
SQL_SERVER = "hant-sql-server"
SQL_DB = "hant-sqldb"
SQL_USER = "sqladmin"
SQL_PASSWORD = "Hant66612!"
JDBC_URL = f"jdbc:sqlserver://{SQL_SERVER}.database.windows.net:1433;database={SQL_DB};encrypt=true;trustServerCertificate=false;loginTimeout=30;"
JDBC_PROPS = {"user": SQL_USER, "password": SQL_PASSWORD,
              "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"}
TARGET_HEADER = "dbo.InvoiceHeader"
TARGET_LINE = "dbo.InvoiceLine"
STG_HEADER = "dbo.InvoiceHeader_Stg"
STG_LINE = "dbo.InvoiceLine_Stg"


def run_sql(sql_text):
    jvm = spark._sc._gateway.jvm
    conn = None
    stmt = None
    try:
        conn = jvm.java.sql.DriverManager.getConnection(
            JDBC_URL, SQL_USER, SQL_PASSWORD)
        stmt = conn.createStatement()
        stmt.execute(sql_text)
    finally:
        stmt.close() if stmt is not None else None
        conn.close() if conn is not None else None


def publish_sql(h, l):
    if h.rdd.isEmpty() or l.rdd.isEmpty():
        return
    h.select("InvoiceId", "OrderDate", "CustomerName", "ShipPostalCode", "ShipCity", "ShipState", "ShipCountry", "ShipMode", "BalanceDue", "SubTotal", "DiscountPercent",
             "DiscountAmount", "ShippingAmount", "InvoiceTotal", "OrderId", "SourceFile").write.mode("overwrite").jdbc(JDBC_URL, STG_HEADER, properties=JDBC_PROPS)
    l.select("InvoiceId", "LineNumber", "ProductName", "SubCategory", "Category", "ProductId", "Quantity", "UnitPrice",
             "ItemSubTotal", "SourceFile").write.mode("overwrite").jdbc(JDBC_URL, STG_LINE, properties=JDBC_PROPS)
    run_sql(f"""IF OBJECT_ID('{TARGET_HEADER}','U') IS NULL CREATE TABLE {TARGET_HEADER}(InvoiceId NVARCHAR(256) NOT NULL PRIMARY KEY,OrderDate DATE NULL,CustomerName NVARCHAR(400) NULL,ShipPostalCode NVARCHAR(50) NULL,ShipCity NVARCHAR(200) NULL,ShipState NVARCHAR(200) NULL,ShipCountry NVARCHAR(200) NULL,ShipMode NVARCHAR(100) NULL,BalanceDue DECIMAL(18,2) NULL,SubTotal DECIMAL(18,2) NULL,DiscountPercent DECIMAL(9,6) NULL,DiscountAmount DECIMAL(18,2) NULL,ShippingAmount DECIMAL(18,2) NULL,InvoiceTotal DECIMAL(18,2) NULL,OrderId NVARCHAR(200) NULL,SourceFile NVARCHAR(1000) NULL); IF OBJECT_ID('{TARGET_LINE}','U') IS NULL CREATE TABLE {TARGET_LINE}(InvoiceId NVARCHAR(256) NOT NULL,LineNumber INT NOT NULL,ProductName NVARCHAR(1000) NULL,SubCategory NVARCHAR(200) NULL,Category NVARCHAR(200) NULL,ProductId NVARCHAR(200) NULL,Quantity DECIMAL(18,4) NULL,UnitPrice DECIMAL(18,4) NULL,ItemSubTotal DECIMAL(18,2) NULL,SourceFile NVARCHAR(1000) NULL,CONSTRAINT PK_InvoiceLine PRIMARY KEY (InvoiceId, LineNumber),CONSTRAINT FK_InvoiceLine_Header FOREIGN KEY (InvoiceId) REFERENCES {TARGET_HEADER}(InvoiceId)); MERGE {TARGET_HEADER} AS tgt USING (SELECT * FROM {STG_HEADER} WHERE InvoiceId IS NOT NULL) AS src ON tgt.InvoiceId=src.InvoiceId WHEN MATCHED THEN UPDATE SET tgt.OrderDate=src.OrderDate,tgt.CustomerName=src.CustomerName,tgt.ShipPostalCode=src.ShipPostalCode,tgt.ShipCity=src.ShipCity,tgt.ShipState=src.ShipState,tgt.ShipCountry=src.ShipCountry,tgt.ShipMode=src.ShipMode,tgt.BalanceDue=src.BalanceDue,tgt.SubTotal=src.SubTotal,tgt.DiscountPercent=src.DiscountPercent,tgt.DiscountAmount=src.DiscountAmount,tgt.ShippingAmount=src.ShippingAmount,tgt.InvoiceTotal=src.InvoiceTotal,tgt.OrderId=src.OrderId,tgt.SourceFile=src.SourceFile WHEN NOT MATCHED THEN INSERT (InvoiceId,OrderDate,CustomerName,ShipPostalCode,ShipCity,ShipState,ShipCountry,ShipMode,BalanceDue,SubTotal,DiscountPercent,DiscountAmount,ShippingAmount,InvoiceTotal,OrderId,SourceFile) VALUES (src.InvoiceId,src.OrderDate,src.CustomerName,src.ShipPostalCode,src.ShipCity,src.ShipState,src.ShipCountry,src.ShipMode,src.BalanceDue,src.SubTotal,src.DiscountPercent,src.DiscountAmount,src.ShippingAmount,src.InvoiceTotal,src.OrderId,src.SourceFile); MERGE {TARGET_LINE} AS tgt USING (SELECT * FROM {STG_LINE} WHERE InvoiceId IS NOT NULL) AS src ON tgt.InvoiceId=src.InvoiceId AND tgt.LineNumber=src.LineNumber WHEN MATCHED THEN UPDATE SET tgt.ProductName=src.ProductName,tgt.SubCategory=src.SubCategory,tgt.Category=src.Category,tgt.ProductId=src.ProductId,tgt.Quantity=src.Quantity,tgt.UnitPrice=src.UnitPrice,tgt.ItemSubTotal=src.ItemSubTotal,tgt.SourceFile=src.SourceFile WHEN NOT MATCHED THEN INSERT (InvoiceId,LineNumber,ProductName,SubCategory,Category,ProductId,Quantity,UnitPrice,ItemSubTotal,SourceFile) VALUES (src.InvoiceId,src.LineNumber,src.ProductName,src.SubCategory,src.Category,src.ProductId,src.Quantity,src.UnitPrice,src.ItemSubTotal,src.SourceFile);""")


ENABLE_SQL_LOAD = True
if ENABLE_SQL_LOAD:
    publish_sql(valid_h, valid_l)
for k, validation in {"gold_header_path": GOLD_HEADER_PATH, "gold_lines_path": GOLD_LINES_PATH, "gold_warning_header_path": GOLD_WARNING_HEADER_PATH, "gold_warning_lines_path": GOLD_WARNING_LINES_PATH, "gold_quarantine_header_path": GOLD_QUARANTINE_HEADER_PATH, "gold_quarantine_lines_path": GOLD_QUARANTINE_LINES_PATH, "gold_issue_log_path": GOLD_ISSUE_LOG_PATH, "gold_run_summary_path": GOLD_RUN_SUMMARY_PATH, "gold_source_publication_summary_path": GOLD_SOURCE_SUMMARY_PATH, "gold_dq_impact_summary_path": GOLD_DQ_IMPACT_PATH, "gold_kpi_recon_summary_path": GOLD_KPI_RECON_PATH}.items():
    dbutils.jobs.taskValues.set(key=k, value=validation)
print("Gold publish finished.")

Gold publish finished.


In [0]:
display(issues)

layer,dataset,rule_id,severity,SourceFile,SourceType,InvoiceId,LineNumber,issue_type,dq_reason,GoldRunId,issue_ts
gold,header,gold_header_shippostalcode_warning,WARNING,null,null,null,null,contract,ShipPostalCode is missing in Gold header,91c89c32-1aef-4b2d-ac16-cf2597f4a108,2026-04-23T19:52:50.161611Z
gold,header,gold_header_shipcity_warning,WARNING,null,null,null,null,contract,ShipCity is missing in Gold header,91c89c32-1aef-4b2d-ac16-cf2597f4a108,2026-04-23T19:52:50.161632Z
gold,header,gold_header_shipstate_warning,WARNING,null,null,null,null,contract,ShipState is missing in Gold header,91c89c32-1aef-4b2d-ac16-cf2597f4a108,2026-04-23T19:52:50.161647Z
gold,lines,gold_line_arith_warning,WARNING,null,null,null,null,line_subtotal_diff,Line subtotal exceeds the strict Gold reconciliation tolerance,91c89c32-1aef-4b2d-ac16-cf2597f4a108,2026-04-23T19:52:50.161696Z
gold,header,gold_header_shipcountry_warning,WARNING,null,null,null,null,contract,ShipCountry is missing in Gold header,91c89c32-1aef-4b2d-ac16-cf2597f4a108,2026-04-23T19:52:50.161661Z
